# Notebook 47: Observability

This notebook demonstrates advanced observability primitives using `multigen.observability`.
All examples use mock implementations — no real API keys are required.

Topics covered:
- `CausalAttributor` — record causal nodes, blame analysis
- `EpistemicDebt` — confidence tracking and propagated uncertainty
- `DecisionAuditTrail` — log decision nodes, render full trail, query flagged
- `CounterfactualReplayer` — replay pipeline with patched node output
- `PatternMiner` — mine recurring workflow patterns from run summaries

In [ ]:
import sys
sys.path.insert(0, '../sdk')


## Mock Implementation of `multigen.observability`

In [ ]:
import statistics
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

print('Imports ready.')


## Section 1 — CausalAttributor

In [ ]:
@dataclass
class CausalNode:
    node_id: str
    output: Any
    upstream: List[str] = field(default_factory=list)  # node_ids this depends on
    confidence: float = 1.0


class CausalAttributor:
    def __init__(self):
        self._nodes: Dict[str, CausalNode] = {}

    def record(self, node_id: str, output: Any, upstream: List[str] = None, confidence: float = 1.0):
        self._nodes[node_id] = CausalNode(
            node_id=node_id, output=output,
            upstream=upstream or [], confidence=confidence
        )

    def blame(self, target_id: str) -> List[Dict]:
        """Return all contributing nodes sorted by blame_score (descending)."""
        visited = set()
        result = []

        def _traverse(nid: str, depth: int):
            if nid in visited or nid not in self._nodes:
                return
            visited.add(nid)
            node = self._nodes[nid]
            # blame_score: higher for direct contributors, weighted by confidence
            blame_score = round(node.confidence / (depth + 1), 4)
            result.append({'node_id': nid, 'depth': depth, 'blame_score': blame_score,
                           'confidence': node.confidence})
            for up in node.upstream:
                _traverse(up, depth + 1)

        _traverse(target_id, 0)
        return sorted(result, key=lambda x: x['blame_score'], reverse=True)


attributor = CausalAttributor()
attributor.record('fetch_data',   output='raw_data',              confidence=0.95)
attributor.record('clean_data',   output='clean_df',  upstream=['fetch_data'], confidence=0.90)
attributor.record('model_infer',  output='predictions', upstream=['clean_data'], confidence=0.75)
attributor.record('report',       output='final_doc', upstream=['model_infer', 'fetch_data'], confidence=0.88)

print('blame("report"):')
for item in attributor.blame('report'):
    print(f'  node={item["node_id"]:15s}  depth={item["depth"]}  blame_score={item["blame_score"]}  confidence={item["confidence"]}')


## Section 2 — EpistemicDebt

In [ ]:
@dataclass
class EpistemicNode:
    node_id: str
    confidence: float
    upstream: List[str] = field(default_factory=list)


@dataclass
class EpistemicReport:
    low_confidence_nodes: List[str]
    propagated_uncertainty: float
    debt_score: float


class EpistemicDebt:
    LOW_CONFIDENCE_THRESHOLD = 0.7

    def __init__(self):
        self._nodes: Dict[str, EpistemicNode] = {}

    def add(self, node_id: str, confidence: float, upstream: List[str] = None):
        self._nodes[node_id] = EpistemicNode(
            node_id=node_id, confidence=confidence, upstream=upstream or []
        )

    def report(self) -> EpistemicReport:
        if not self._nodes:
            return EpistemicReport([], 0.0, 0.0)

        low_conf = [
            n.node_id for n in self._nodes.values()
            if n.confidence < self.LOW_CONFIDENCE_THRESHOLD
        ]

        # Propagated uncertainty: product of (1 - confidence) for all nodes
        total_uncertainty = 1.0
        for n in self._nodes.values():
            total_uncertainty *= (1.0 - n.confidence)
        prop_uncertainty = round(total_uncertainty, 6)

        # Debt score: mean low-confidence weight
        all_conf = [n.confidence for n in self._nodes.values()]
        debt_score = round(1.0 - statistics.mean(all_conf), 4)

        return EpistemicReport(
            low_confidence_nodes=low_conf,
            propagated_uncertainty=prop_uncertainty,
            debt_score=debt_score,
        )


ed = EpistemicDebt()
ed.add('retrieve',  confidence=0.95)
ed.add('parse',     confidence=0.85, upstream=['retrieve'])
ed.add('infer',     confidence=0.60, upstream=['parse'])       # low
ed.add('summarise', confidence=0.65, upstream=['infer'])       # low
ed.add('format',    confidence=0.92, upstream=['summarise'])

rpt = ed.report()
print(f'low_confidence_nodes  : {rpt.low_confidence_nodes}')
print(f'propagated_uncertainty: {rpt.propagated_uncertainty}')
print(f'debt_score            : {rpt.debt_score}')


## Section 3 — DecisionAuditTrail

In [ ]:
@dataclass
class AuditNode:
    step: int
    node_id: str
    decision: str
    rationale: str
    confidence: float = 1.0
    flagged: bool = False


class DecisionAuditTrail:
    LOW_CONF = 0.7

    def __init__(self):
        self._trail: List[AuditNode] = []

    def log(self, node_id: str, decision: str, rationale: str,
            confidence: float = 1.0, flag: bool = False):
        step = len(self._trail) + 1
        self._trail.append(AuditNode(
            step=step, node_id=node_id, decision=decision,
            rationale=rationale, confidence=confidence, flagged=flag
        ))

    def render(self) -> str:
        lines = ['=== Decision Audit Trail ===']
        for n in self._trail:
            flag_marker = ' [FLAG]' if n.flagged else ''
            conf_marker = ' [LOW-CONF]' if n.confidence < self.LOW_CONF else ''
            lines.append(
                f'Step {n.step:02d} | {n.node_id:20s} | conf={n.confidence:.2f}{conf_marker}{flag_marker}'
            )
            lines.append(f'         Decision  : {n.decision}')
            lines.append(f'         Rationale : {n.rationale}')
        return '\n'.join(lines)

    def query(self, flagged: bool = False, low_confidence: bool = False) -> List[AuditNode]:
        result = self._trail
        if flagged:
            result = [n for n in result if n.flagged]
        if low_confidence:
            result = [n for n in result if n.confidence < self.LOW_CONF]
        return result


trail = DecisionAuditTrail()
trail.log('route_intent',   'SEARCH',           'User asked a factual question',   confidence=0.95)
trail.log('retrieve_docs',  'VECTOR_SEARCH',    'Embedding similarity query',       confidence=0.88)
trail.log('rank_results',   'TOP_5_SELECTED',   'BM25 reranking applied',           confidence=0.80)
trail.log('generate_answer','GPT4_COMPLETION',  'Prompt length within limit',       confidence=0.65, flag=True)
trail.log('validate_output','PASS',             'No policy violations detected',    confidence=0.72)

print(trail.render())

print('\nFlagged nodes:')
for n in trail.query(flagged=True):
    print(f'  {n.node_id}: {n.decision}')

print('\nLow-confidence nodes:')
for n in trail.query(low_confidence=True):
    print(f'  {n.node_id}: conf={n.confidence}')


## Section 4 — CounterfactualReplayer

In [ ]:
@dataclass
class ShadowRunResult:
    original_output: Any
    counterfactual_output: Any
    changed: bool


class CounterfactualReplayer:
    """
    Replay a pipeline defined as a list of (node_id, fn) pairs.
    A patch can override the output of a specific node.
    """

    def __init__(self, pipeline: List[Tuple[str, Callable]]):
        self.pipeline = pipeline  # [(node_id, fn), ...]

    def run(self, initial_input: Any, patches: Dict[str, Any] = None) -> Dict[str, Any]:
        patches = patches or {}
        ctx = initial_input
        trace = {}
        for node_id, fn in self.pipeline:
            if node_id in patches:
                ctx = patches[node_id]
                trace[node_id] = ('patched', ctx)
            else:
                ctx = fn(ctx)
                trace[node_id] = ('real', ctx)
        return {'output': ctx, 'trace': trace}

    def replay(self, initial_input: Any, patch_node: str, patch_value: Any) -> ShadowRunResult:
        original = self.run(initial_input)
        counterfactual = self.run(initial_input, patches={patch_node: patch_value})
        return ShadowRunResult(
            original_output=original['output'],
            counterfactual_output=counterfactual['output'],
            changed=(original['output'] != counterfactual['output']),
        )


# Simple three-node pipeline
def tokenise(text: str) -> List[str]:
    return text.lower().split()

def score_tokens(tokens: List[str]) -> float:
    positive = {'good', 'great', 'excellent', 'happy'}
    return round(sum(t in positive for t in tokens) / max(len(tokens), 1), 3)

def classify(score: float) -> str:
    return 'positive' if score >= 0.2 else 'negative'


pipeline = [('tokenise', tokenise), ('score', score_tokens), ('classify', classify)]
replayer = CounterfactualReplayer(pipeline)

# Original run
original_result = replayer.run('This is a good and great day')
print('Original run:')
for node_id, (kind, val) in original_result['trace'].items():
    print(f'  {node_id}: ({kind}) {val}')

# Counterfactual: patch the score node to output a low score
cf = replayer.replay('This is a good and great day',
                     patch_node='score', patch_value=0.05)
print(f'\nCounterfactual replay (patching "score" -> 0.05):')
print(f'  original_output      : {cf.original_output}')
print(f'  counterfactual_output: {cf.counterfactual_output}')
print(f'  changed              : {cf.changed}')

# Counterfactual that does NOT change the outcome
cf2 = replayer.replay('This is a good and great day',
                      patch_node='tokenise', patch_value=['this', 'is', 'great'])
print(f'\nCounterfactual replay (patching "tokenise" -> same positive tokens):')
print(f'  original_output      : {cf2.original_output}')
print(f'  counterfactual_output: {cf2.counterfactual_output}')
print(f'  changed              : {cf2.changed}')


## Section 5 — PatternMiner

In [ ]:
@dataclass
class WorkflowRunSummary:
    run_id: str
    node_sequence: List[str]   # ordered list of node_ids executed
    outcome: float             # final outcome score


@dataclass
class MiningResult:
    pattern: Tuple[str, ...]
    count: int
    mean_outcome: float


class PatternMiner:
    def __init__(self):
        self._runs: List[WorkflowRunSummary] = []

    def add(self, run: WorkflowRunSummary):
        self._runs.append(run)

    def mine(self, seq_length: int = 2) -> List[MiningResult]:
        pattern_outcomes: Dict[Tuple, List[float]] = defaultdict(list)

        for run in self._runs:
            seq = run.node_sequence
            for i in range(len(seq) - seq_length + 1):
                pattern = tuple(seq[i: i + seq_length])
                pattern_outcomes[pattern].append(run.outcome)

        results = [
            MiningResult(
                pattern=pat,
                count=len(outcomes),
                mean_outcome=round(statistics.mean(outcomes), 4),
            )
            for pat, outcomes in pattern_outcomes.items()
        ]
        return sorted(results, key=lambda r: (r.count, r.mean_outcome), reverse=True)


miner = PatternMiner()

runs_data = [
    ('r01', ['A','B','C','D'], 0.85),
    ('r02', ['A','B','C','E'], 0.78),
    ('r03', ['A','B','D','E'], 0.72),
    ('r04', ['A','B','C','D'], 0.90),
    ('r05', ['X','B','C','D'], 0.65),
    ('r06', ['A','B','C','D'], 0.88),
    ('r07', ['A','C','D','E'], 0.70),
    ('r08', ['X','B','C','E'], 0.60),
    ('r09', ['A','B','C','D'], 0.82),
    ('r10', ['A','B','D'],     0.75),
]

for run_id, seq, outcome in runs_data:
    miner.add(WorkflowRunSummary(run_id=run_id, node_sequence=seq, outcome=outcome))

print('Top patterns (seq_length=2):')
for r in miner.mine(seq_length=2)[:8]:
    print(f'  {"->" .join(r.pattern):15s}  count={r.count}  mean_outcome={r.mean_outcome}')

print('\nTop patterns (seq_length=3):')
for r in miner.mine(seq_length=3)[:6]:
    print(f'  {"->" .join(r.pattern):20s}  count={r.count}  mean_outcome={r.mean_outcome}')
